# Final Heat-Map Computer Vision Models

UNSW-NB15 training and synthetic IPv6 external evaluation for ResNet50, EfficientNet-B0, and HeatWaveNet heat-map models.


In [ ]:
import os
import json
import math
import random
import glob
import zipfile
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.utils.class_weight import compute_class_weight

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from torchvision import models

GLOBAL_SEED = 42
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(GLOBAL_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

RUN_TASKS = ["binary", "multiclass"]    # or ["binary"] / ["multiclass"]
RUN_MODELS = ["resnet50_heatmap", "efficientnetb0_heatmap", "heatwavenet_novel"]
RUN_EXPLAINABILITY = True
RUN_SHAP = False

KAGGLE_INPUT_ROOT = Path("/kaggle/input") if Path("/kaggle/input").exists() else None

def find_file(patterns):
    if KAGGLE_INPUT_ROOT is not None:
        for pat in patterns:
            hits = glob.glob(str(KAGGLE_INPUT_ROOT / "**" / pat), recursive=True)
            if hits:
                return hits[0]
    for pat in patterns:
        hits = glob.glob(str(Path("/mnt/data") / "**" / pat), recursive=True)
        if hits:
            return hits[0]
    return None

UNSW_TRAIN_CSV = find_file(["UNSW_NB15_training-set.csv"])
UNSW_TEST_CSV = find_file(["UNSW_NB15_testing-set.csv"])
SYN_ZIP_PATH = find_file(["synthetic_ipv6_grounded_v3_32x32.zip"])

USE_HF_DOWNLOAD = False
if (UNSW_TRAIN_CSV is None or UNSW_TEST_CSV is None):
    USE_HF_DOWNLOAD = True

OUT_ROOT = Path("final_model_artifacts")
for p in [OUT_ROOT, OUT_ROOT / "combined", OUT_ROOT / "cache"]:
    p.mkdir(parents=True, exist_ok=True)

BASE_SIDE = 32
MODEL_IMAGE_SIZE = 128
VAL_SIZE = 0.15
BATCH_SIZE_TRAIN = 64
BATCH_SIZE_EVAL = 128
TUNE_EPOCHS = 4
FINAL_EPOCHS = 12
PATIENCE = 3
MAX_ROWS_UNSW = None   # set e.g. 50000 for a faster dry run
NUM_WORKERS = 2

print("UNSW train csv:", UNSW_TRAIN_CSV)
print("UNSW test csv :", UNSW_TEST_CSV)
print("Synthetic zip :", SYN_ZIP_PATH)
print("Output root   :", OUT_ROOT.resolve())


## Dataset Loading

UNSW-NB15 official split and synthetic IPv6 train/validation/test split files.


In [ ]:
if USE_HF_DOWNLOAD:
    from huggingface_hub import hf_hub_download
    HF_REPO = "Mouwiya/UNSW-NB15-small"
    UNSW_TRAIN_CSV = hf_hub_download(repo_id=HF_REPO, filename="UNSW_NB15_training-set.csv", repo_type="dataset")
    UNSW_TEST_CSV  = hf_hub_download(repo_id=HF_REPO, filename="UNSW_NB15_testing-set.csv",  repo_type="dataset")

assert UNSW_TRAIN_CSV is not None and UNSW_TEST_CSV is not None, "UNSW CSV files not found."
assert SYN_ZIP_PATH is not None, "Synthetic ZIP file not found."

unsw_train = pd.read_csv(UNSW_TRAIN_CSV)
unsw_test  = pd.read_csv(UNSW_TEST_CSV)

with zipfile.ZipFile(SYN_ZIP_PATH, "r") as zf:
    names = zf.namelist()
    syn_root = sorted(set([n.split("/")[0] for n in names if "/" in n]))[0]
    syn_flows = pd.read_csv(zf.open(f"{syn_root}/flows.csv"))
    syn_train_split = pd.read_csv(zf.open(f"{syn_root}/train.csv"))
    syn_val_split   = pd.read_csv(zf.open(f"{syn_root}/val.csv"))
    syn_test_split  = pd.read_csv(zf.open(f"{syn_root}/test.csv"))

syn_train = syn_flows.merge(syn_train_split[["record_id"]], on="record_id", how="inner")
syn_val   = syn_flows.merge(syn_val_split[["record_id"]],   on="record_id", how="inner")
syn_test  = syn_flows.merge(syn_test_split[["record_id"]],  on="record_id", how="inner")

if MAX_ROWS_UNSW is not None and len(unsw_train) > MAX_ROWS_UNSW:
    keep_frac = MAX_ROWS_UNSW / len(unsw_train)
    unsw_train = (
        unsw_train.groupby("attack_cat", group_keys=False)
        .apply(lambda g: g.sample(max(1, int(round(len(g) * keep_frac))), random_state=GLOBAL_SEED))
        .reset_index(drop=True)
    )

print("UNSW train/test:", unsw_train.shape, unsw_test.shape)
print("Synthetic splits:", len(syn_train), len(syn_val), len(syn_test), "total:", len(syn_flows))
display(unsw_train.head(2))
display(syn_flows.head(2))


## Helpers


In [ ]:
def normalize_attack_name(x):
    if pd.isna(x):
        return "Unknown"
    s = str(x).strip()
    if s.lower() == "normal":
        return "Benign"
    return s

def save_json(obj, path):
    Path(path).write_text(json.dumps(obj, indent=2))

def save_text(text, path):
    Path(path).write_text(str(text))

def plot_bar(series, title, xlabel, ylabel, save_path, top_k=None):
    s = series.copy()
    if top_k is not None:
        s = s.head(top_k)
    plt.figure(figsize=(10, 4))
    s.plot(kind="bar")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(save_path, dpi=180, bbox_inches="tight")
    plt.show()

def plot_confusion(cm, labels, title, save_path, max_labels=20):
    show_labels = labels[:max_labels]
    show_cm = np.array(cm)[:max_labels, :max_labels]
    plt.figure(figsize=(8, 6))
    plt.imshow(show_cm)
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.colorbar()
    ticks = np.arange(len(show_labels))
    plt.xticks(ticks, show_labels, rotation=45, ha="right")
    plt.yticks(ticks, show_labels)
    plt.tight_layout()
    plt.savefig(save_path, dpi=180, bbox_inches="tight")
    plt.show()

def plot_curves(history, save_prefix):
    epochs = list(range(1, len(history["train_loss"]) + 1))
    plt.figure(figsize=(8, 4))
    plt.plot(epochs, history["train_loss"], label="train_loss")
    plt.title("Training loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{save_prefix}_train_loss.png", dpi=180, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(8, 4))
    plt.plot(epochs, history["val_acc"], label="val_acc")
    plt.plot(epochs, history["val_primary"], label="val_primary_metric")
    plt.title("Validation metrics")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{save_prefix}_val_metrics.png", dpi=180, bbox_inches="tight")
    plt.show()

def multiclass_metrics(y_true, y_pred, y_prob, labels):
    acc = accuracy_score(y_true, y_pred)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", labels=labels, zero_division=0
    )
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", labels=labels, zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    out = {
        "accuracy": float(acc),
        "macro_precision": float(p_macro),
        "macro_recall": float(r_macro),
        "macro_f1": float(f1_macro),
        "weighted_precision": float(p_weighted),
        "weighted_recall": float(r_weighted),
        "weighted_f1": float(f1_weighted),
        "confusion_matrix": cm.tolist(),
        "report": classification_report(y_true, y_pred, labels=labels, zero_division=0, digits=4),
    }
    try:
        y_true_idx = pd.Series(y_true).map({l:i for i,l in enumerate(labels)}).values
        out["ovr_auc"] = float(roc_auc_score(y_true_idx, y_prob, multi_class="ovr", average="macro"))
    except Exception:
        out["ovr_auc"] = None
    return out

def binary_metrics(y_true, y_pred, y_score):
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", pos_label="Attack", zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=["Benign", "Attack"])
    out = {
        "accuracy": float(acc),
        "precision": float(p),
        "recall": float(r),
        "f1": float(f1),
        "confusion_matrix": cm.tolist(),
        "report": classification_report(y_true, y_pred, labels=["Benign", "Attack"], digits=4, zero_division=0),
    }
    try:
        out["auc"] = float(roc_auc_score((pd.Series(y_true) == "Attack").astype(int).values, y_score))
    except Exception:
        out["auc"] = None
    return out

def bootstrap_ci(metric_fn, y_true, y_pred, n_boot=500, alpha=0.95, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    vals = []
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        vals.append(metric_fn(y_true[idx], y_pred[idx]))
    lo = np.quantile(vals, (1 - alpha) / 2)
    hi = np.quantile(vals, 1 - (1 - alpha) / 2)
    return float(lo), float(hi)

def binary_f1_metric(y_true, y_pred):
    return precision_recall_fscore_support(y_true, y_pred, average="binary", pos_label="Attack", zero_division=0)[2]

def multiclass_macro_f1_metric(y_true, y_pred):
    labels = sorted(pd.Series(y_true).unique().tolist())
    return precision_recall_fscore_support(y_true, y_pred, average="macro", labels=labels, zero_division=0)[2]


## Task Preparation

Binary and multiclass tasks. Synthetic IPv6 is used as the external evaluation set.


In [ ]:
def prepare_unsw_task(task):
    tr = unsw_train.copy()
    te = unsw_test.copy()

    if task == "binary":
        target = "label"
        tr[target] = tr[target].astype(str).map(lambda x: "Attack" if str(x) == "1" else "Benign")
        te[target] = te[target].astype(str).map(lambda x: "Attack" if str(x) == "1" else "Benign")
        labels = ["Benign", "Attack"]
        drop_cols = ["id", "attack_cat"]
    else:
        target = "attack_cat"
        tr[target] = tr[target].apply(normalize_attack_name)
        te[target] = te[target].apply(normalize_attack_name)
        labels = sorted(pd.Series(tr[target]).unique().tolist())
        drop_cols = ["id", "label"]

    X_train_full = tr.drop(columns=[target] + drop_cols, errors="ignore").reset_index(drop=True)
    y_train_full = tr[target].astype(str).values
    X_test = te.drop(columns=[target] + drop_cols, errors="ignore").reset_index(drop=True)
    y_test = te[target].astype(str).values

    return X_train_full, y_train_full, X_test, y_test, labels

def prepare_synthetic_task(task):
    te = syn_test.copy()
    if task == "binary":
        te["label"] = te["label"].astype(str).map(lambda x: "Benign" if x == "Benign" else "Attack")
        labels = ["Benign", "Attack"]
    else:
        te["label"] = te["label"].astype(str).apply(lambda s: "Benign" if str(s).lower() == "normal" else str(s))
        labels = sorted(pd.Series(te["label"]).unique().tolist())

    X_test = te.drop(columns=["label", "record_id", "window_start_utc", "window_end_utc", "src_ip", "dst_ip"], errors="ignore").reset_index(drop=True)
    y_test = te["label"].astype(str).values
    return X_test, y_test, labels

for task in RUN_TASKS:
    Xu, yu, Xu_test, yu_test, labels = prepare_unsw_task(task)
    Xs, ys, syn_labels = prepare_synthetic_task(task)
    print(task, "| UNSW train/test:", Xu.shape, Xu_test.shape, "| Synthetic test:", Xs.shape)


## Heat-Map Encoder

Flow-level tabular rows are encoded as 3-channel heat-map images.


In [ ]:
class HeatmapTabularEncoder:
    def __init__(self, base_side=32):
        self.base_side = base_side
        self.top_k = base_side * base_side
        self.preprocess = None
        self.selected_idx = None
        self.sel_min = None
        self.sel_max = None
        self.cat_cols = None
        self.num_cols = None

    def fit(self, X_train):
        self.cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object"]
        self.num_cols = [c for c in X_train.columns if c not in self.cat_cols]

        self.preprocess = ColumnTransformer(
            transformers=[
                ("num", Pipeline([
                    ("imp", SimpleImputer(strategy="median")),
                    ("sc", StandardScaler(with_mean=True)),
                ]), self.num_cols),
                ("cat", Pipeline([
                    ("imp", SimpleImputer(strategy="most_frequent")),
                    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse=False)),
                ]), self.cat_cols),
            ],
            remainder="drop"
        )
        Xv = np.asarray(self.preprocess.fit_transform(X_train), dtype=np.float32)
        vars_ = Xv.var(axis=0)
        top_k = min(self.top_k, Xv.shape[1])
        idx = np.sort(np.argsort(vars_)[-top_k:])
        self.selected_idx = idx
        Xs = Xv[:, self.selected_idx]
        self.sel_min = Xs.min(axis=0)
        self.sel_max = Xs.max(axis=0)
        self.sel_max = np.where(self.sel_max == self.sel_min, self.sel_min + 1e-6, self.sel_max)
        return self

    def transform_vectors(self, X):
        Xv = np.asarray(self.preprocess.transform(X), dtype=np.float32)
        Xs = Xv[:, self.selected_idx]
        return Xs.astype(np.float32)

    def vectors_to_images(self, Xs):
        x1 = (Xs - self.sel_min) / (self.sel_max - self.sel_min + 1e-8)
        x1 = np.clip(x1, 0, 1)
        x2 = np.abs(np.diff(x1, axis=1, prepend=x1[:, :1]))
        ranks = np.argsort(np.argsort(Xs, axis=1), axis=1).astype(np.float32)
        denom = max(Xs.shape[1] - 1, 1)
        x3 = ranks / float(denom)

        target = self.top_k
        def pad_to_target(x):
            if x.shape[1] < target:
                pad = np.zeros((len(x), target - x.shape[1]), dtype=np.float32)
                return np.concatenate([x, pad], axis=1)
            return x[:, :target]

        x1 = pad_to_target(x1).reshape(len(x1), 1, self.base_side, self.base_side)
        x2 = pad_to_target(x2).reshape(len(x2), 1, self.base_side, self.base_side)
        x3 = pad_to_target(x3).reshape(len(x3), 1, self.base_side, self.base_side)
        return np.concatenate([x1, x2, x3], axis=1).astype(np.float32)


## Cached Heat-Map Tensors


In [ ]:
def build_cached_task_data(task):
    cache_dir = OUT_ROOT / "cache" / task
    cache_dir.mkdir(parents=True, exist_ok=True)

    cache_meta = cache_dir / "meta.json"
    if cache_meta.exists():
        print(f"Loading cached tensors for task={task}")
        meta = json.loads(cache_meta.read_text())
        return {
            "X_train_img": np.load(cache_dir / "X_train_img.npy"),
            "X_val_img": np.load(cache_dir / "X_val_img.npy"),
            "X_unsw_test_img": np.load(cache_dir / "X_unsw_test_img.npy"),
            "X_syn_test_img": np.load(cache_dir / "X_syn_test_img.npy"),
            "y_train": np.load(cache_dir / "y_train.npy"),
            "y_val": np.load(cache_dir / "y_val.npy"),
            "y_unsw_test": np.load(cache_dir / "y_unsw_test.npy"),
            "y_syn_test": np.load(cache_dir / "y_syn_test.npy"),
            "labels": meta["labels"],
            "syn_eval_mask_kept": meta["syn_eval_mask_kept"],
        }

    X_train_full, y_train_full, X_unsw_test, y_unsw_test_lbl, labels = prepare_unsw_task(task)
    X_syn_test, y_syn_test_lbl, _ = prepare_synthetic_task(task)

    X_train, X_val, y_train_lbl, y_val_lbl = train_test_split(
        X_train_full, y_train_full, test_size=VAL_SIZE, random_state=GLOBAL_SEED, stratify=y_train_full
    )

    syn_keep = pd.Series(y_syn_test_lbl).isin(labels).values
    X_syn_eval = X_syn_test.loc[syn_keep].reset_index(drop=True)
    y_syn_eval_lbl = pd.Series(y_syn_test_lbl)[syn_keep].reset_index(drop=True).values

    label2id = {l:i for i,l in enumerate(labels)}
    y_train = pd.Series(y_train_lbl).map(label2id).astype(int).values
    y_val = pd.Series(y_val_lbl).map(label2id).astype(int).values
    y_unsw_test = pd.Series(y_unsw_test_lbl).map(label2id).astype(int).values
    y_syn_test = pd.Series(y_syn_eval_lbl).map(label2id).astype(int).values

    encoder = HeatmapTabularEncoder(base_side=BASE_SIDE).fit(X_train)
    X_train_img = encoder.vectors_to_images(encoder.transform_vectors(X_train))
    X_val_img = encoder.vectors_to_images(encoder.transform_vectors(X_val))
    X_unsw_test_img = encoder.vectors_to_images(encoder.transform_vectors(X_unsw_test))
    X_syn_test_img = encoder.vectors_to_images(encoder.transform_vectors(X_syn_eval))

    np.save(cache_dir / "X_train_img.npy", X_train_img)
    np.save(cache_dir / "X_val_img.npy", X_val_img)
    np.save(cache_dir / "X_unsw_test_img.npy", X_unsw_test_img)
    np.save(cache_dir / "X_syn_test_img.npy", X_syn_test_img)
    np.save(cache_dir / "y_train.npy", y_train)
    np.save(cache_dir / "y_val.npy", y_val)
    np.save(cache_dir / "y_unsw_test.npy", y_unsw_test)
    np.save(cache_dir / "y_syn_test.npy", y_syn_test)

    meta = {
        "labels": labels,
        "syn_eval_mask_kept": int(syn_keep.sum()),
        "syn_total_rows": int(len(y_syn_test_lbl)),
    }
    save_json(meta, cache_meta)

    plt.figure(figsize=(8, 3))
    for i in range(3):
        plt.subplot(1, 3, i+1)
        plt.imshow(np.transpose(X_train_img[0], (1,2,0))[:,:,i])
        plt.axis("off")
        plt.title(f"Channel {i+1}")
    plt.suptitle(f"Heat-map example ({task})")
    plt.tight_layout()
    plt.savefig(cache_dir / "heatmap_example.png", dpi=180, bbox_inches="tight")
    plt.show()

    return {
        "X_train_img": X_train_img,
        "X_val_img": X_val_img,
        "X_unsw_test_img": X_unsw_test_img,
        "X_syn_test_img": X_syn_test_img,
        "y_train": y_train,
        "y_val": y_val,
        "y_unsw_test": y_unsw_test,
        "y_syn_test": y_syn_test,
        "labels": labels,
        "syn_eval_mask_kept": int(syn_keep.sum()),
    }

TASK_CACHE = {}
for task in RUN_TASKS:
    TASK_CACHE[task] = build_cached_task_data(task)
    print(task, "cached:", TASK_CACHE[task]["X_train_img"].shape, TASK_CACHE[task]["X_syn_test_img"].shape)


## Image Datasets and Losses


In [ ]:
class HeatmapImageDataset(Dataset):
    def __init__(self, x, y, img_size=128):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.img_size = img_size

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        img = self.x[idx]
        if img.shape[-1] != self.img_size:
            img = F.interpolate(img.unsqueeze(0), size=(self.img_size, self.img_size), mode="bilinear", align_corners=False).squeeze(0)
        return img, self.y[idx]

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha, reduction="none")
        pt = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean()

def make_loaders(x_train, y_train, x_val, y_val, img_size=128, batch_train=64, batch_eval=128):
    class_counts = np.bincount(y_train, minlength=len(np.unique(y_train)))
    sample_weights = 1.0 / np.maximum(class_counts[y_train], 1)
    train_loader = DataLoader(
        HeatmapImageDataset(x_train, y_train, img_size=img_size),
        batch_size=batch_train,
        sampler=WeightedRandomSampler(torch.tensor(sample_weights, dtype=torch.double), len(sample_weights), replacement=True),
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    val_loader = DataLoader(
        HeatmapImageDataset(x_val, y_val, img_size=img_size),
        batch_size=batch_eval,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    return train_loader, val_loader


## Final Models


In [ ]:
class ResNet50Heatmap(nn.Module):
    def __init__(self, num_classes, dropout=0.3, pretrained=True):
        super().__init__()
        try:
            weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
            self.backbone = models.resnet50(weights=weights)
        except Exception:
            self.backbone = models.resnet50(weights=None)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_features, num_classes))
        self.cam_target = self.backbone.layer4[-1].conv3

    def forward(self, x):
        return self.backbone(x)

class EfficientNetB0Heatmap(nn.Module):
    def __init__(self, num_classes, dropout=0.3, pretrained=True):
        super().__init__()
        try:
            weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
            self.backbone = models.efficientnet_b0(weights=weights)
        except Exception:
            self.backbone = models.efficientnet_b0(weights=None)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_features, num_classes))
        self.cam_target = self.backbone.features[-1]

    def forward(self, x):
        return self.backbone(x)

class SEBlock(nn.Module):
    def __init__(self, channels, r=8):
        super().__init__()
        hidden = max(channels // r, 4)
        self.net = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, hidden, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.net(x)

class SpatialGate(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, 1, kernel_size=7, padding=3)

    def forward(self, x):
        gate = torch.sigmoid(self.conv(x))
        return x * gate

class MultiScaleSEBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.proj = nn.Identity()
        if in_ch != out_ch or stride != 1:
            self.proj = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

        branch_ch = out_ch // 3
        rem = out_ch - 2 * branch_ch

        self.b1 = nn.Sequential(
            nn.Conv2d(in_ch, branch_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(branch_ch),
            nn.ReLU(inplace=True),
        )
        self.b2 = nn.Sequential(
            nn.Conv2d(in_ch, branch_ch, 5, stride=stride, padding=2, bias=False),
            nn.BatchNorm2d(branch_ch),
            nn.ReLU(inplace=True),
        )
        self.b3 = nn.Sequential(
            nn.Conv2d(in_ch, rem, 3, stride=stride, padding=2, dilation=2, bias=False),
            nn.BatchNorm2d(rem),
            nn.ReLU(inplace=True),
        )
        self.mix = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.se = SEBlock(out_ch)
        self.act = nn.ReLU(inplace=True)
        self.out_conv_for_cam = self.mix[0]

    def forward(self, x):
        identity = self.proj(x)
        y = torch.cat([self.b1(x), self.b2(x), self.b3(x)], dim=1)
        y = self.mix(y)
        y = self.se(y)
        y = y + identity
        return self.act(y)

class GeMPool(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p), kernel_size=(x.size(-2), x.size(-1))).pow(1.0 / self.p)

class HeatWaveNet(nn.Module):
    def __init__(self, num_classes, base_channels=32, dropout=0.3):
        super().__init__()
        c = base_channels
        self.stem = nn.Sequential(
            nn.Conv2d(3, c, 3, padding=1, bias=False),
            nn.BatchNorm2d(c),
            nn.ReLU(inplace=True),
        )
        self.stage1 = MultiScaleSEBlock(c, c)
        self.stage2 = MultiScaleSEBlock(c, c * 2, stride=2)
        self.stage3 = MultiScaleSEBlock(c * 2, c * 4, stride=2)
        self.spatial = SpatialGate(c * 4)
        self.stage4 = MultiScaleSEBlock(c * 4, c * 4)
        self.gem = GeMPool()
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(c * 4, num_classes)
        self.cam_target = self.stage4.out_conv_for_cam

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.spatial(x)
        x = self.stage4(x)
        x = self.gem(x).flatten(1)
        x = self.drop(x)
        return self.fc(x)

def set_trainable_backbone(model, model_name, trainable_backbone=True):
    if model_name == "heatwavenet_novel":
        return
    for p in model.backbone.parameters():
        p.requires_grad = trainable_backbone

def build_model(model_name, num_classes, config):
    pretrained = config.get("pretrained", True)
    dropout = config.get("dropout", 0.3)
    if model_name == "resnet50_heatmap":
        model = ResNet50Heatmap(num_classes=num_classes, dropout=dropout, pretrained=pretrained)
        set_trainable_backbone(model, model_name, config.get("trainable_backbone", False))
        return model
    if model_name == "efficientnetb0_heatmap":
        model = EfficientNetB0Heatmap(num_classes=num_classes, dropout=dropout, pretrained=pretrained)
        set_trainable_backbone(model, model_name, config.get("trainable_backbone", False))
        return model
    if model_name == "heatwavenet_novel":
        return HeatWaveNet(num_classes=num_classes, base_channels=config.get("base_channels", 32), dropout=dropout)
    raise ValueError(model_name)


## Hyperparameter Search Spaces


In [ ]:
MODEL_GRIDS = {
    "resnet50_heatmap": list(ParameterGrid({
        "pretrained": [True],
        "dropout": [0.2, 0.4],
        "lr": [3e-4, 1e-4],
        "weight_decay": [1e-4],
        "loss_name": ["ce", "focal"],
        "trainable_backbone": [False, True],
    })),
    "efficientnetb0_heatmap": list(ParameterGrid({
        "pretrained": [True],
        "dropout": [0.2, 0.4],
        "lr": [3e-4, 1e-4],
        "weight_decay": [1e-4],
        "loss_name": ["ce", "focal"],
        "trainable_backbone": [False, True],
    })),
    "heatwavenet_novel": list(ParameterGrid({
        "dropout": [0.2, 0.35],
        "base_channels": [32, 48],
        "lr": [1e-3, 5e-4],
        "weight_decay": [1e-4, 1e-5],
        "loss_name": ["ce", "focal"],
    })),
}
for k, v in MODEL_GRIDS.items():
    print(k, "configs:", len(v))


## Training and Evaluation Functions


In [ ]:
def get_loss_fn(loss_name, class_weights):
    alpha = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
    if loss_name == "focal":
        return FocalLoss(alpha=alpha, gamma=2.0)
    return nn.CrossEntropyLoss(weight=alpha)

def predict_loader(model, loader):
    model.eval()
    ys, preds, probs = [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            logits = model(xb)
            pr = torch.softmax(logits, dim=1).cpu().numpy()
            pdx = pr.argmax(axis=1)
            ys.append(yb.numpy())
            preds.append(pdx)
            probs.append(pr)
    return np.concatenate(ys), np.concatenate(preds), np.concatenate(probs)

def primary_metric(task, y_true_idx, y_pred_idx):
    if task == "binary":
        y_true_lbl = np.where(y_true_idx == 1, "Attack", "Benign")
        y_pred_lbl = np.where(y_pred_idx == 1, "Attack", "Benign")
        return binary_f1_metric(y_true_lbl, y_pred_lbl)
    return precision_recall_fscore_support(y_true_idx, y_pred_idx, average="macro", zero_division=0)[2]

def train_one_model(model_name, task, config, data_bundle, out_dir, run_name, epochs):
    num_classes = len(data_bundle["labels"])
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(num_classes),
        y=data_bundle["y_train"]
    ).astype(np.float32)

    train_loader, val_loader = make_loaders(
        data_bundle["X_train_img"], data_bundle["y_train"],
        data_bundle["X_val_img"], data_bundle["y_val"],
        img_size=MODEL_IMAGE_SIZE, batch_train=BATCH_SIZE_TRAIN, batch_eval=BATCH_SIZE_EVAL
    )

    model = build_model(model_name, num_classes, config).to(DEVICE)
    loss_fn = get_loss_fn(config["loss_name"], class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    scaler = GradScaler(enabled=torch.cuda.is_available())

    history = {"train_loss": [], "val_acc": [], "val_primary": []}
    best_val = -1
    bad = 0
    model_path = out_dir / f"{run_name}.pt"

    for epoch in range(1, epochs + 1):
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=torch.cuda.is_available()):
                logits = model(xb)
                loss = loss_fn(logits, yb)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            losses.append(loss.item())

        yv_true, yv_pred, _ = predict_loader(model, val_loader)
        val_acc = accuracy_score(yv_true, yv_pred)
        val_primary = primary_metric(task, yv_true, yv_pred)

        history["train_loss"].append(float(np.mean(losses)))
        history["val_acc"].append(float(val_acc))
        history["val_primary"].append(float(val_primary))

        print(f"{run_name} | epoch {epoch:02d} | loss={np.mean(losses):.4f} | val_acc={val_acc:.4f} | val_primary={val_primary:.4f}")

        if val_primary > best_val:
            best_val = val_primary
            bad = 0
            torch.save(model.state_dict(), model_path)
        else:
            bad += 1
            if bad >= PATIENCE:
                print("Early stopping.")
                break

    plot_curves(history, str(out_dir / run_name))
    save_json(history, out_dir / f"{run_name}_history.json")

    model = build_model(model_name, num_classes, config).to(DEVICE)
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    return model, history, best_val, model_path

def tune_model(task, model_name, data_bundle, model_task_dir):
    rows = []
    best = None
    best_score = -1

    trial_dir = model_task_dir / "tuning"
    trial_dir.mkdir(parents=True, exist_ok=True)

    for i, cfg in enumerate(MODEL_GRIDS[model_name], start=1):
        run_name = f"trial_{i:02d}"
        model, history, score, model_path = train_one_model(
            model_name=model_name,
            task=task,
            config=cfg,
            data_bundle=data_bundle,
            out_dir=trial_dir,
            run_name=run_name,
            epochs=TUNE_EPOCHS,
        )
        row = {"trial": i, **cfg, "best_val_primary": float(score)}
        rows.append(row)
        if score > best_score:
            best_score = score
            best = cfg.copy()

    tuning_df = pd.DataFrame(rows).sort_values("best_val_primary", ascending=False)
    tuning_df.to_csv(model_task_dir / "tuning_results.csv", index=False)
    save_json(best, model_task_dir / "best_config.json")
    display(tuning_df)
    return best, tuning_df

def evaluate_model(task, model_name, model, data_bundle, out_dir, labels):
    unsw_test_loader = DataLoader(
        HeatmapImageDataset(data_bundle["X_unsw_test_img"], data_bundle["y_unsw_test"], img_size=MODEL_IMAGE_SIZE),
        batch_size=BATCH_SIZE_EVAL, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
    )
    syn_test_loader = DataLoader(
        HeatmapImageDataset(data_bundle["X_syn_test_img"], data_bundle["y_syn_test"], img_size=MODEL_IMAGE_SIZE),
        batch_size=BATCH_SIZE_EVAL, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
    )

    y_true_unsw_idx, y_pred_unsw_idx, y_prob_unsw = predict_loader(model, unsw_test_loader)
    y_true_syn_idx, y_pred_syn_idx, y_prob_syn = predict_loader(model, syn_test_loader)

    id2label = {i:l for i,l in enumerate(labels)}
    y_true_unsw_lbl = pd.Series(y_true_unsw_idx).map(id2label).values
    y_pred_unsw_lbl = pd.Series(y_pred_unsw_idx).map(id2label).values
    y_true_syn_lbl = pd.Series(y_true_syn_idx).map(id2label).values
    y_pred_syn_lbl = pd.Series(y_pred_syn_idx).map(id2label).values

    if task == "binary":
        unsw_score = y_prob_unsw[:, 1]
        syn_score = y_prob_syn[:, 1]
        m_unsw = binary_metrics(y_true_unsw_lbl, y_pred_unsw_lbl, unsw_score)
        m_syn = binary_metrics(y_true_syn_lbl, y_pred_syn_lbl, syn_score)
        lo, hi = bootstrap_ci(binary_f1_metric, y_true_syn_lbl, y_pred_syn_lbl)
        m_syn["f1_ci95"] = [lo, hi]
    else:
        m_unsw = multiclass_metrics(y_true_unsw_lbl, y_pred_unsw_lbl, y_prob_unsw, labels)
        m_syn = multiclass_metrics(y_true_syn_lbl, y_pred_syn_lbl, y_prob_syn, labels)
        lo, hi = bootstrap_ci(multiclass_macro_f1_metric, y_true_syn_lbl, y_pred_syn_lbl)
        m_syn["macro_f1_ci95"] = [lo, hi]

    save_json(m_unsw, out_dir / "unsw_test_metrics.json")
    save_json(m_syn, out_dir / "synthetic_test_metrics.json")
    save_text(m_unsw["report"], out_dir / "unsw_test_report.txt")
    save_text(m_syn["report"], out_dir / "synthetic_test_report.txt")

    if task == "binary":
        cm_labels = ["Benign", "Attack"]
        pd.DataFrame(m_unsw["confusion_matrix"], index=cm_labels, columns=cm_labels).to_csv(out_dir / "unsw_confusion_matrix.csv")
        pd.DataFrame(m_syn["confusion_matrix"], index=cm_labels, columns=cm_labels).to_csv(out_dir / "synthetic_confusion_matrix.csv")
        plot_confusion(m_unsw["confusion_matrix"], cm_labels, f"{model_name} | UNSW test confusion", out_dir / "unsw_confusion.png", max_labels=2)
        plot_confusion(m_syn["confusion_matrix"], cm_labels, f"{model_name} | Synthetic confusion", out_dir / "synthetic_confusion.png", max_labels=2)
    else:
        pd.DataFrame(m_unsw["confusion_matrix"], index=labels, columns=labels).to_csv(out_dir / "unsw_confusion_matrix.csv")
        pd.DataFrame(m_syn["confusion_matrix"], index=labels, columns=labels).to_csv(out_dir / "synthetic_confusion_matrix.csv")
        plot_confusion(m_unsw["confusion_matrix"], labels, f"{model_name} | UNSW test confusion", out_dir / "unsw_confusion.png", max_labels=15)
        plot_confusion(m_syn["confusion_matrix"], labels, f"{model_name} | Synthetic confusion", out_dir / "synthetic_confusion.png", max_labels=15)

    pd.DataFrame({"y_true": y_true_unsw_lbl, "y_pred": y_pred_unsw_lbl}).to_csv(out_dir / "unsw_predictions.csv", index=False)
    pd.DataFrame({"y_true": y_true_syn_lbl, "y_pred": y_pred_syn_lbl}).to_csv(out_dir / "synthetic_predictions.csv", index=False)

    return m_unsw, m_syn


## Tuning and Final Training


In [ ]:
ALL_RESULTS = []

for task in RUN_TASKS:
    data_bundle = TASK_CACHE[task]
    labels = data_bundle["labels"]

    base_task_dir = OUT_ROOT / task
    base_task_dir.mkdir(parents=True, exist_ok=True)

    plot_bar(pd.Series(data_bundle["y_train"]).map({i:l for i,l in enumerate(labels)}).value_counts().sort_values(ascending=False),
             f"{task}: UNSW train label distribution (used for final models)",
             "Class", "Count",
             base_task_dir / "unsw_train_label_distribution.png", top_k=15)

    plot_bar(pd.Series(data_bundle["y_syn_test"]).map({i:l for i,l in enumerate(labels)}).value_counts().sort_values(ascending=False),
             f"{task}: Synthetic evaluation label distribution",
             "Class", "Count",
             base_task_dir / "synthetic_label_distribution.png", top_k=15)

    for model_name in RUN_MODELS:
        print("=" * 90)
        print("TASK:", task, "| MODEL:", model_name)

        model_task_dir = OUT_ROOT / task / model_name
        model_task_dir.mkdir(parents=True, exist_ok=True)

        best_cfg, tuning_df = tune_model(task, model_name, data_bundle, model_task_dir)

        final_dir = model_task_dir / "final_run"
        final_dir.mkdir(parents=True, exist_ok=True)

        best_model, history, best_val, model_path = train_one_model(
            model_name=model_name,
            task=task,
            config=best_cfg,
            data_bundle=data_bundle,
            out_dir=final_dir,
            run_name=f"{model_name}_best",
            epochs=FINAL_EPOCHS,
        )

        m_unsw, m_syn = evaluate_model(task, model_name, best_model, data_bundle, final_dir, labels)

        row = {
            "task": task,
            "model": model_name,
            "best_val_primary": best_val,
            **{f"cfg_{k}": v for k, v in best_cfg.items()},
        }

        if task == "binary":
            row.update({
                "unsw_accuracy": m_unsw["accuracy"],
                "unsw_precision": m_unsw["precision"],
                "unsw_recall": m_unsw["recall"],
                "unsw_f1": m_unsw["f1"],
                "unsw_auc": m_unsw["auc"],
                "synthetic_accuracy": m_syn["accuracy"],
                "synthetic_precision": m_syn["precision"],
                "synthetic_recall": m_syn["recall"],
                "synthetic_f1": m_syn["f1"],
                "synthetic_auc": m_syn["auc"],
                "synthetic_f1_ci95_lo": m_syn["f1_ci95"][0],
                "synthetic_f1_ci95_hi": m_syn["f1_ci95"][1],
            })
        else:
            row.update({
                "unsw_accuracy": m_unsw["accuracy"],
                "unsw_macro_precision": m_unsw["macro_precision"],
                "unsw_macro_recall": m_unsw["macro_recall"],
                "unsw_macro_f1": m_unsw["macro_f1"],
                "unsw_weighted_f1": m_unsw["weighted_f1"],
                "unsw_ovr_auc": m_unsw["ovr_auc"],
                "synthetic_accuracy": m_syn["accuracy"],
                "synthetic_macro_precision": m_syn["macro_precision"],
                "synthetic_macro_recall": m_syn["macro_recall"],
                "synthetic_macro_f1": m_syn["macro_f1"],
                "synthetic_weighted_f1": m_syn["weighted_f1"],
                "synthetic_ovr_auc": m_syn["ovr_auc"],
                "synthetic_macro_f1_ci95_lo": m_syn["macro_f1_ci95"][0],
                "synthetic_macro_f1_ci95_hi": m_syn["macro_f1_ci95"][1],
            })

        ALL_RESULTS.append(row)

final_results_df = pd.DataFrame(ALL_RESULTS)
final_results_df.to_csv(OUT_ROOT / "combined" / "final_model_results.csv", index=False)
display(final_results_df)


## Final Model Comparison Tables


In [ ]:
results_df = pd.read_csv(OUT_ROOT / "combined" / "final_model_results.csv")

binary_df = results_df[results_df["task"] == "binary"].copy()
multiclass_df = results_df[results_df["task"] == "multiclass"].copy()

binary_cols = [
    "model", "unsw_accuracy", "unsw_precision", "unsw_recall", "unsw_f1", "unsw_auc",
    "synthetic_accuracy", "synthetic_precision", "synthetic_recall", "synthetic_f1", "synthetic_auc",
    "synthetic_f1_ci95_lo", "synthetic_f1_ci95_hi"
]
multiclass_cols = [
    "model", "unsw_accuracy", "unsw_macro_precision", "unsw_macro_recall", "unsw_macro_f1", "unsw_weighted_f1", "unsw_ovr_auc",
    "synthetic_accuracy", "synthetic_macro_precision", "synthetic_macro_recall", "synthetic_macro_f1", "synthetic_weighted_f1", "synthetic_ovr_auc",
    "synthetic_macro_f1_ci95_lo", "synthetic_macro_f1_ci95_hi"
]

if len(binary_df) > 0:
    binary_cmp = binary_df[binary_cols].sort_values("synthetic_f1", ascending=False)
    binary_cmp.to_csv(OUT_ROOT / "combined" / "binary_final_model_comparison.csv", index=False)
    print("Binary final model comparison")
    display(binary_cmp)

if len(multiclass_df) > 0:
    multiclass_cmp = multiclass_df[multiclass_cols].sort_values("synthetic_macro_f1", ascending=False)
    multiclass_cmp.to_csv(OUT_ROOT / "combined" / "multiclass_final_model_comparison.csv", index=False)
    print("Multiclass final model comparison")
    display(multiclass_cmp)

best_rows = []
if len(binary_df) > 0:
    best_rows.append(binary_df.sort_values("synthetic_f1", ascending=False).iloc[0].to_dict())
if len(multiclass_df) > 0:
    best_rows.append(multiclass_df.sort_values("synthetic_macro_f1", ascending=False).iloc[0].to_dict())

best_df = pd.DataFrame(best_rows)
best_df.to_csv(OUT_ROOT / "combined" / "best_final_model_per_task.csv", index=False)
print("Best final model per task")
display(best_df)


## Optional Benchmark Merge


In [ ]:
bench_candidates = [
    Path("benchmark_artifacts/combined/benchmark_comparison_all.csv"),
    Path("cnn_benchmark_artifacts/combined/unsw_benchmark_cnn_comparison.csv"),
]
existing = [p for p in bench_candidates if p.exists()]
if existing:
    print("Found benchmark files:")
    for p in existing:
        print("-", p)
else:
    print("No prior benchmark CSVs found in working directory. Skipping merge.")


## Grad-CAM


In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self._register()

    def _register(self):
        def forward_hook(module, inp, out):
            self.activations = out.detach()
        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()
        self.fh = self.target_layer.register_forward_hook(forward_hook)
        self.bh = self.target_layer.register_full_backward_hook(backward_hook)

    def remove(self):
        self.fh.remove()
        self.bh.remove()

    def __call__(self, x, class_idx=None):
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)
        if class_idx is None:
            class_idx = logits.argmax(dim=1).item()
        score = logits[:, class_idx].sum()
        score.backward()
        grads = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (grads * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(x.shape[-2], x.shape[-1]), mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

def overlay_cam(img, cam):
    base = np.transpose(img, (1,2,0))
    base = (base - base.min()) / (base.max() - base.min() + 1e-8)
    heat = plt.cm.jet(cam)[..., :3]
    overlay = 0.45 * base + 0.55 * heat
    return np.clip(overlay, 0, 1)

if RUN_EXPLAINABILITY:
    best_df = pd.read_csv(OUT_ROOT / "combined" / "best_final_model_per_task.csv")
    for _, row in best_df.iterrows():
        task = row["task"]
        model_name = row["model"]
        labels = TASK_CACHE[task]["labels"]
        num_classes = len(labels)

        cfg = json.loads((OUT_ROOT / task / model_name / "best_config.json").read_text())
        model_path = OUT_ROOT / task / model_name / "final_run" / f"{model_name}_best.pt"
        model = build_model(model_name, num_classes, cfg).to(DEVICE)
        model.load_state_dict(torch.load(model_path, map_location=DEVICE))
        model.eval()

        explain_dir = OUT_ROOT / task / model_name / "explainability"
        explain_dir.mkdir(parents=True, exist_ok=True)

        x = TASK_CACHE[task]["X_syn_test_img"]
        y = TASK_CACHE[task]["y_syn_test"]

        cam_engine = GradCAM(model, model.cam_target)

        for idx in range(min(4, len(x))):
            img = torch.tensor(x[idx:idx+1], dtype=torch.float32)
            img = F.interpolate(img, size=(MODEL_IMAGE_SIZE, MODEL_IMAGE_SIZE), mode="bilinear", align_corners=False).to(DEVICE)
            cam = cam_engine(img)
            pred = model(img).argmax(dim=1).item()
            overlay = overlay_cam(img.squeeze(0).cpu().numpy(), cam)

            plt.figure(figsize=(6, 3))
            plt.subplot(1, 2, 1)
            plt.imshow(np.transpose(img.squeeze(0).cpu().numpy(), (1,2,0)))
            plt.title(f"Input | true={labels[y[idx]]}")
            plt.axis("off")

            plt.subplot(1, 2, 2)
            plt.imshow(overlay)
            plt.title(f"Grad-CAM | pred={labels[pred]}")
            plt.axis("off")

            plt.tight_layout()
            plt.savefig(explain_dir / f"gradcam_sample_{idx}.png", dpi=180, bbox_inches="tight")
            plt.show()

        cam_engine.remove()


## Optional SHAP


In [ ]:
if RUN_SHAP:
    try:
        import shap
        best_df = pd.read_csv(OUT_ROOT / "combined" / "best_final_model_per_task.csv")
        row = best_df.iloc[0]
        task = row["task"]
        model_name = row["model"]
        labels = TASK_CACHE[task]["labels"]
        num_classes = len(labels)

        cfg = json.loads((OUT_ROOT / task / model_name / "best_config.json").read_text())
        model = build_model(model_name, num_classes, cfg).to(DEVICE)
        model.load_state_dict(torch.load(OUT_ROOT / task / model_name / "final_run" / f"{model_name}_best.pt", map_location=DEVICE))
        model.eval()

        x_bg = TASK_CACHE[task]["X_train_img"][:20]
        x_explain = TASK_CACHE[task]["X_syn_test_img"][:5]

        x_bg = torch.tensor(x_bg, dtype=torch.float32)
        x_bg = F.interpolate(x_bg, size=(MODEL_IMAGE_SIZE, MODEL_IMAGE_SIZE), mode="bilinear", align_corners=False).to(DEVICE)
        x_explain = torch.tensor(x_explain, dtype=torch.float32)
        x_explain = F.interpolate(x_explain, size=(MODEL_IMAGE_SIZE, MODEL_IMAGE_SIZE), mode="bilinear", align_corners=False).to(DEVICE)

        explainer = shap.GradientExplainer(model, x_bg)
        shap_values = explainer.shap_values(x_explain)

        shap_dir = OUT_ROOT / task / model_name / "explainability"
        shap_dir.mkdir(parents=True, exist_ok=True)
        np.save(shap_dir / "shap_values.npy", np.array(shap_values, dtype=object))
        print("SHAP values saved.")
    except Exception as e:
        print("SHAP run skipped due to:", e)


## Artifact Dashboard


In [ ]:
def list_files(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted([p.name for p in folder.glob("*")])

dashboard = {}
for task in RUN_TASKS:
    dashboard[task] = {}
    for model_name in RUN_MODELS:
        base = OUT_ROOT / task / model_name
        dashboard[task][model_name] = {
            "root": list_files(base),
            "tuning": list_files(base / "tuning"),
            "final_run": list_files(base / "final_run"),
            "explainability": list_files(base / "explainability"),
        }

dashboard["combined"] = list_files(OUT_ROOT / "combined")
save_json(dashboard, OUT_ROOT / "combined" / "artefact_dashboard.json")
dashboard


## Result Summary Export


In [ ]:
lines = []
lines.append("# Final Model Results Summary")
lines.append("")
lines.append("## Completed Runs")
lines.append("- Built final heat-map computer vision models.")
lines.append("- Tuned all models only on the UNSW validation split.")
lines.append("- Evaluated in-domain on UNSW official test.")
lines.append("- Evaluated externally on the synthetic dataset.")
lines.append("- Exported comparison tables, confusion matrices, reports, and XAI artefacts.")
lines.append("")
lines.append("## Final models")
lines.append("- resnet50_heatmap")
lines.append("- efficientnetb0_heatmap")
lines.append("- heatwavenet_novel")
lines.append("")
lines.append("## Key files")
lines.append("- combined/final_model_results.csv")
lines.append("- combined/binary_final_model_comparison.csv")
lines.append("- combined/multiclass_final_model_comparison.csv")
lines.append("- combined/best_final_model_per_task.csv")
lines.append("- combined/artefact_dashboard.json")
lines.append("")
lines.append("## Thesis use")
lines.append("- Use UNSW test results to show benchmark-level in-domain performance.")
lines.append("- Use synthetic results to discuss the true project evaluation setting.")
lines.append("- Use Grad-CAM outputs to support the interpretability discussion.")
lines.append("- Use the novel model results to justify the thesis contribution beyond the benchmark architectures.")
summary_path = OUT_ROOT / "combined" / "final_model_results_summary.md"
save_text("\n".join(lines), summary_path)
print("Saved:", summary_path.resolve())
